# Dataset information

NASA data set, obtained from a series of aerodynamic and acoustic tests of two and three-dimensional airfoil blade sections conducted in an anechoic wind tunnel.

Feature Type: Real
Instances: 1503
Features: 5

The NASA data set comprises different size NACA 0012 airfoils at various wind tunnel speeds and angles of attack. The span of the airfoil and the observer position were the same in all of the experiments. 

https://archive.ics.uci.edu/dataset/291/airfoil%2Bself-noise 

# Download dataset

In [1]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
ds = fetch_ucirepo(id=291) 
  
# data (as pandas dataframes) 
X = ds.data.features 
y = ds.data.targets 
  
# metadata 
print(ds.metadata) 
  
# variable information 
print(ds.variables) 


{'uci_id': 291, 'name': 'Airfoil Self-Noise', 'repository_url': 'https://archive.ics.uci.edu/dataset/291/airfoil+self+noise', 'data_url': 'https://archive.ics.uci.edu/static/public/291/data.csv', 'abstract': 'NASA data set, obtained from a series of aerodynamic and acoustic tests of two and three-dimensional airfoil blade sections conducted in an anechoic wind tunnel.', 'area': 'Physics and Chemistry', 'tasks': ['Regression'], 'characteristics': ['Multivariate'], 'num_instances': 1503, 'num_features': 5, 'feature_types': ['Real'], 'demographics': [], 'target_col': ['scaled-sound-pressure'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 1989, 'last_updated': 'Fri Mar 29 2024', 'dataset_doi': '10.24432/C5VW2C', 'creators': ['Thomas Brooks', 'D. Pope', 'Michael Marcolini'], 'intro_paper': None, 'additional_info': {'summary': 'The NASA data set comprises different size NACA 0012 airfoils at various wind tunnel speeds and angles of

In [3]:
ds.data.keys()

dict_keys(['ids', 'features', 'targets', 'original', 'headers'])

In [5]:
df = ds.data.original
df

,frequency,attack-angle,chord-length,free-stream-velocity,suction-side-displacement-thickness,scaled-sound-pressure
0,800,0.0,0.3048,71.3,0.002663,126.201
1,1000,0.0,0.3048,71.3,0.002663,125.201
2,1250,0.0,0.3048,71.3,0.002663,125.951
3,1600,0.0,0.3048,71.3,0.002663,127.591
4,2000,0.0,0.3048,71.3,0.002663,127.461
...,...,...,...,...,...,...
1498,2500,15.6,0.1016,39.6,0.052849,110.264
1499,3150,15.6,0.1016,39.6,0.052849,109.254
1500,4000,15.6,0.1016,39.6,0.052849,106.604
1501,5000,15.6,0.1016,39.6,0.052849,106.224


In [8]:
ds.keys()

dict_keys(['data', 'metadata', 'variables'])

In [10]:
ds.variables

,name,role,type,demographic,description,units,missing_values
0,frequency,Feature,Integer,None,None,Hz,no
1,attack-angle,Feature,Binary,None,None,deg,no
2,chord-length,Feature,Continuous,None,None,m,no
3,free-stream-velocity,Feature,Continuous,None,None,m/s,no
4,suction-side-displacement-thickness,Feature,Continuous,None,None,m,no
5,scaled-sound-pressure,Target,Continuous,None,None,dB,no


# Predicting the noise (scaled-sound-pressure)

In [11]:
import torch
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

# 1. Daten aufteilen: 80 % zum Lernen, 20 % für den abschließenden Test.
# X und y stammen aus der Download-Zelle oben.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 2. Werte skalieren, damit das Netz leichter lernen kann.
# fit_transform lernt Mittelwert und Streuung NUR aus den Trainingsdaten.
x_scaler = StandardScaler()
y_scaler = StandardScaler()
X_train_scaled = x_scaler.fit_transform(X_train)
y_train_scaled = y_scaler.fit_transform(y_train)

# PyTorch rechnet mit Tensoren statt mit DataFrames.
# y_train_tensor hat die Form (Anzahl Beispiele, 1), passend zum Netzausgang.
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_scaled, dtype=torch.float32)

# 3. Kleines Netz: 5 Features -> 32 Neuronen -> 16 Neuronen -> 1 Schallpegel.
# ReLU ermöglicht nichtlineare Zusammenhänge; der Ausgang bleibt linear.
torch.manual_seed(42)
model = nn.Sequential(
    nn.Linear(5, 32),
    nn.ReLU(),
    nn.Linear(32, 16),
    nn.ReLU(),
    nn.Linear(16, 1),
)
loss_function = nn.MSELoss()  # Mittlerer quadratischer Fehler
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# 4. TRAINING: In jeder Epoche verwenden wir alle Trainingsbeispiele gemeinsam.
model.train()
for epoch in range(1000):
    prediction = model(X_train_tensor)                  # Vorhersage berechnen
    loss = loss_function(prediction, y_train_tensor)    # Fehler berechnen
    optimizer.zero_grad()                              # Alte Gradienten löschen
    loss.backward()                                    # Neue Gradienten berechnen
    optimizer.step()                                   # Gewichte anpassen

# 5. TEST: Nur die bisher unbenutzten Testdaten auswerten.
# transform verwendet die bereits gelernte Skalierung der Trainingsdaten.
X_test_scaled = x_scaler.transform(X_test)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

model.eval()
with torch.no_grad():  # Beim Test brauchen wir keine Gradienten.
    prediction_test_scaled = model(X_test_tensor)

# Vorhersagen aus der skalierten Darstellung wieder in dB umrechnen.
prediction_test = y_scaler.inverse_transform(prediction_test_scaled.numpy())
mae = mean_absolute_error(y_test, prediction_test)
print(f"Mittlerer absoluter Fehler auf den Testdaten: {mae:.2f} dB")


Mittlerer absoluter Fehler auf den Testdaten: 1.17 dB


In [12]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

# Baseline: lineare Regression mit allen fünf Features.
# Wir verwenden dieselbe Aufteilung wie beim MLP in der vorherigen Zelle.
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)  # Nur auf den Trainingsdaten lernen.

# Auswertung ausschließlich auf den Testdaten, wieder in dB.
prediction_linear = linear_model.predict(X_test)
mae_linear = mean_absolute_error(y_test, prediction_linear)

# Je kleiner der mittlere absolute Fehler, desto besser.
print(f"Lineare Regression – Testfehler: {mae_linear:.2f} dB")
print(f"PyTorch-MLP        – Testfehler: {mae:.2f} dB")


Lineare Regression – Testfehler: 3.67 dB
PyTorch-MLP        – Testfehler: 1.17 dB
